In [69]:
# Import data handling libraries
import pandas as pd  
import numpy as np  

# Import deep learning framework
import tensorflow as tf  
from tensorflow import keras  

# Import layers for building neural networks
from tensorflow.keras.layers import Embedding, LSTM, Dense  

# Import Sequential model (not used later)
from tensorflow.keras.models import Sequential  

In [71]:
# Load dataset from CSV file
train = pd.read_csv("engtamilTrain.csv")  

# Remove unwanted column (usually auto-generated index)
train = train.drop(["Unnamed: 0"], axis=1)  

In [72]:
# Extract English sentences column
english_sentences = train["en"]  

# Extract Tamil sentences column
tamil_sentence = train['ta']  

In [93]:
# Limit dataset to first 1000 rows (for faster training)
english_sentences = english_sentences.head(1500)  
tamil_sentences = tamil_sentence.head(1500)  

In [95]:
def addSosEos(seriesSentence):
    # Define the <SOS> and <EOS> tokens
    sos_token = "<SOS>"
    eos_token = "<EOS>"

    # Add <SOS> and <EOS> tokens to each statement
    statements_with_tokens = [f"{sos_token} {statement} {eos_token}" for statement in seriesSentence]

    english_sent=[]
    # Print the statements with tokens
    for statement in statements_with_tokens:
        english_sent.append(statement)
        print(statement)
    return english_sent


In [97]:
# Apply function to both languages
english_sent_SE=addSosEos(english_sentences)
tamil_sent_SE=addSosEos(tamil_sentences)

<SOS> MMA vice president Qazi Hussain Ahmad declared last month: 'We are not extremists.
 <EOS>
<SOS> Information has surfaced in recent years suggesting that Julius Rosenberg was involved in passing some form of intelligence to Soviet officials during the Second World War.
 <EOS>
<SOS> And Azor begat Sadoc; and Sadoc begat Achim; and Achim begat Eliud;
 <EOS>
<SOS> She says she knows what is going on, but can do nothing about it.
 <EOS>
<SOS> And be it indeed that I have erred, my error remains with myself.
 <EOS>
<SOS> Finally, the columnist fails to tell us who among the political leaders of the bourgeoisie, past and present, he counts among the paragons of morality.
 <EOS>
<SOS> These include the British Tamil Forum, La Maison du Tamil Eelam (France), the Canadian Tamil Congress, and the Swiss Tamil Forum.
 <EOS>
<SOS> Vijay accompanied with his wife and daughter enjoyed the film 'Anjathey'.
 <EOS>
<SOS> Both Musharraf and Vajpayee have exploited the current war drive to divert pub

In [99]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [101]:
# Tokenize the English and Tamil sentences
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(english_sent_SE)
english_vocab_size = len(english_tokenizer.word_index) + 1
english_sequences = english_tokenizer.texts_to_sequences(english_sent_SE)

In [103]:
tamil_tokenizer = Tokenizer(filters="")
tamil_tokenizer.fit_on_texts(tamil_sent_SE)
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)

In [105]:
max_input_seq_length=20
max_output_seq_length=20

In [107]:
# Pad sequences to a fixed length
input_sequences = pad_sequences(english_sequences, maxlen=max_input_seq_length, padding='post')
output_sequences = pad_sequences(tamil_sequences, maxlen=max_output_seq_length, padding='post')

In [109]:
# Prepare the decoder input and output sequences for teacher forcing
decoder_input_sequences = np.zeros_like(output_sequences)
decoder_input_sequences[:, 1:] = output_sequences[:, :-1]
decoder_input_sequences[:, 0] = tamil_tokenizer.word_index['<sos>']

decoder_output_sequences = np.eye(tamil_vocab_size)[output_sequences]


In [110]:
from gensim.models import Word2Vec

eng_model = Word2Vec.load('engmodel.bin')
tam_model = Word2Vec.load('tammodel.bin')


In [112]:
def create_embedding_matrix(word2vec_model, tokenizer, vocab_size):
    embedding_matrix = np.zeros((vocab_size, word2vec_model.vector_size))
    for word, i in tokenizer.word_index.items():
        try:
            embedding_vector = word2vec_model.wv[word]
            embedding_matrix[i] = embedding_vector
        except KeyError:
            pass  # Words not found in the embedding index will be all zeros
    return embedding_matrix

eng_embedding_matrix = create_embedding_matrix(eng_model, english_tokenizer, english_vocab_size)
tam_embedding_matrix = create_embedding_matrix(tam_model, tamil_tokenizer, tamil_vocab_size)


In [114]:
eng_embedding_matrix.shape

(9201, 100)

In [117]:
tam_embedding_matrix.shape

(13791, 100)

In [119]:
def create_seq2seq_model(input_vocab_size, output_vocab_size, input_seq_length, output_seq_length, hidden_units, eng_embedding_matrix, tam_embedding_matrix):
    # Encoder
    encoder_inputs = Input(shape=(input_seq_length,))
    encoder_embedding = Embedding(input_vocab_size, hidden_units, weights=[eng_embedding_matrix], trainable=False)(encoder_inputs)
    encoder_lstm, encoder_state_h, encoder_state_c = LSTM(hidden_units, return_state=True)(encoder_embedding)

    # Decoder
    decoder_inputs = Input(shape=(output_seq_length,))
    decoder_embedding = Embedding(output_vocab_size, hidden_units, weights=[tam_embedding_matrix], trainable=False)(decoder_inputs)
    decoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[encoder_state_h, encoder_state_c])
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_outputs)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    return model


In [121]:
# Convert target_sequences to one-hot encoded format
target_sequences = tf.keras.utils.to_categorical(output_sequences, num_classes=tamil_vocab_size)


In [123]:
model = create_seq2seq_model(english_vocab_size, tamil_vocab_size, max_input_seq_length, max_output_seq_length, 100, eng_embedding_matrix, tam_embedding_matrix)

In [125]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [127]:
# Fit the model to the data
batch_size = 32
epochs = 50
model.fit([input_sequences, output_sequences], decoder_output_sequences, batch_size=batch_size, epochs=epochs, validation_split=0.2)


Epoch 1/50
38/38 [==============================] - 18s 249ms/step - loss: 8.9793 - accuracy: 0.2589 - val_loss: 7.9072 - val_accuracy: 0.2708
Epoch 2/50
38/38 [==============================] - 7s 174ms/step - loss: 6.9800 - accuracy: 0.2678 - val_loss: 7.4217 - val_accuracy: 0.2708
Epoch 3/50
38/38 [==============================] - 7s 173ms/step - loss: 6.5760 - accuracy: 0.2684 - val_loss: 7.5287 - val_accuracy: 0.2708
Epoch 4/50
38/38 [==============================] - 6s 171ms/step - loss: 6.4579 - accuracy: 0.2684 - val_loss: 7.4952 - val_accuracy: 0.2703
Epoch 5/50
38/38 [==============================] - 6s 168ms/step - loss: 6.3756 - accuracy: 0.2686 - val_loss: 7.5843 - val_accuracy: 0.2712
Epoch 6/50
38/38 [==============================] - 6s 168ms/step - loss: 6.3158 - accuracy: 0.2695 - val_loss: 7.6193 - val_accuracy: 0.2715
Epoch 7/50
38/38 [==============================] - 6s 170ms/step - loss: 6.2656 - accuracy: 0.2696 - val_loss: 7.6691 - val_accuracy: 0.2715
Epoch

In [139]:
# Preprocessing the input
input_sentence ="<sos>Information has surfaced in recent years suggesting that Julius Rosenberg was involved in passing some form of intelligence to Soviet officials during the Second World War<eos>"
input_sentencea = "<sos>Finally, the columnist fails to tell us who among the political leaders of the bourgeoisie, past and present, he counts among the paragons of morality<eos>"
#input_sentence = "<sos> Hello <eos>"
# Convert the input sentence to sequence
input_sequence = english_tokenizer.texts_to_sequences([input_sentence])

# Pad the statement to the maximum input sequence length
input_sequence = pad_sequences(input_sequence, maxlen=max_input_seq_length, padding='post')

# Generate predictions
predictions = model.predict([input_sequence, np.zeros((1, max_output_seq_length))])

# Convert predictions to tokens
predicted_tokens = np.argmax(predictions, axis=-1)[0]

# Create index to word mapping for Tamil vocabulary
tamil_index_word = {i: w for w, i in tamil_tokenizer.word_index.items()}


# Convert tokens to text
decoded_sentence = []
for token in predicted_tokens:
    if token == 0:  # Assuming 0 is the padding token
        continue
    word = tamil_index_word.get(token)
    if word == '<eos>':
        break
    if word is not None:
        decoded_sentence.append(word)
    else:
        decoded_sentence.append('<unk>')

# Join the words to form the decoded statement
decoded_statement = ' '.join(decoded_sentence)

# Print the decoded statement
print(decoded_statement)



1/1 [==============================] - 0s 30ms/step
<sos> ஒரு மற்றும் ஒரு ஒரு ஒரு ஒரு ஒரு ஒரு


In [131]:
print(translate_sentence("Finally, the columnist fails to tell us the truth"))

1/1 [==============================] - 0s 46ms/step
இந்த இந்த சென்னை படங்களில்


In [141]:
print(translate_sentence("Hello"))
print(translate_sentence("I am fine"))
print(translate_sentence("Where are you"))

print(translate_sentence("i will leave"))

1/1 [==============================] - 0s 26ms/step

1/1 [==============================] - 0s 28ms/step

1/1 [==============================] - 0s 28ms/step

1/1 [==============================] - 0s 28ms/step

